# MultiDecoderDPRNN Baseline Floor Evaluation

This notebook runs the pretrained **MultiDecoderDPRNN** (unknown speaker count, 2-5 speakers) backend to separate both **Libri2Mix** and **Libri3Mix** test sets and score them using our PIT evaluation harness.

### Kaggle Setup:
* **Accelerator:** GPU (T4 x2 or P100)
* **Internet:** On
* **Attached Utility Files:** Ensure `separate.py`, `evaluate.py`, and `make_libri3mix_test.py` are uploaded as a utility dataset and attached.

In [ ]:
!pip -q install asteroid speechbrain torchaudio 2>/dev/null
import torch
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device:', torch.cuda.get_device_name(0))

### 1. Locate Source Scripts

In [ ]:
import sys, os, glob
hits = glob.glob('/kaggle/input/**/evaluate.py', recursive=True)
assert hits, 'evaluate.py not found under /kaggle/input -- is the dataset attached?'
SRC_DIR = os.path.dirname(hits[0])
sys.path.insert(0, SRC_DIR)
print('Using source scripts from:', SRC_DIR)
import separate as S, evaluate as E

### 2. Download LibriSpeech and Generate Libri2Mix + Libri3Mix Test Sets
We clone JorisCos/LibriMix to fetch metadata and download LibriSpeech `test-clean` (~350 MB) to dynamically generate the mixtures.

In [ ]:
%cd /kaggle/working
!wget -q -c https://www.openslr.org/resources/12/test-clean.tar.gz
!tar -xzf test-clean.tar.gz
!git clone -q https://github.com/JorisCos/LibriMix
print('Source audio and metadata download complete.')

In [ ]:
# Generate 2-speaker test set (min, 8kHz)
print('--- Generating Libri2Mix Test Set ---')
!python {SRC_DIR}/make_libri3mix_test.py \
    --metadata LibriMix/metadata/Libri2Mix/libri2mix_test-clean.csv \
    --librispeech-root LibriSpeech \
    --outdir libri2mix_test --sr 8000

# Generate 3-speaker test set (min, 8kHz)
print('\n--- Generating Libri3Mix Test Set ---')
!python {SRC_DIR}/make_libri3mix_test.py \
    --metadata LibriMix/metadata/Libri3Mix/libri3mix_test-clean.csv \
    --librispeech-root LibriSpeech \
    --outdir libri3mix_test --sr 8000

### 3. Evaluate on 2 Speakers (Libri2Mix)

In [ ]:
DATA_ROOT_2 = 'libri2mix_test'
META_2      = 'libri2mix_test/manifest_test.csv'
N_SRC_2     = 2
MODEL       = 'JunzheJosephZhu/MultiDecoderDPRNN'
OUT_2       = '/kaggle/working/est_2spk'

print('Loading model and initializing backend for 2 speakers...')
backend = S.build_backend(MODEL, 'cuda:0' if torch.cuda.is_available() else 'cpu')

print('Running separation on 2-speaker mixtures...')
S.separate_librimix(backend, META_2, DATA_ROOT_2, OUT_2, limit=None)

print('Evaluating 2-speaker separation quality...')
rows_2 = E._rows_from_librimix(META_2, DATA_ROOT_2, OUT_2, N_SRC_2)
df_2 = E.evaluate_manifest(rows_2)
print(f'\nEvaluated {len(df_2)} mixtures')
print(E.summarize(df_2).to_string(index=False))

### 4. Evaluate on 3 Speakers (Libri3Mix)

In [ ]:
DATA_ROOT_3 = 'libri3mix_test'
META_3      = 'libri3mix_test/manifest_test.csv'
N_SRC_3     = 3
OUT_3       = '/kaggle/working/est_3spk'

# Reuse the same loaded backend
print('Running separation on 3-speaker mixtures...')
S.separate_librimix(backend, META_3, DATA_ROOT_3, OUT_3, limit=None)

print('Evaluating 3-speaker separation quality...')
rows_3 = E._rows_from_librimix(META_3, DATA_ROOT_3, OUT_3, N_SRC_3)
df_3 = E.evaluate_manifest(rows_3)
print(f'\nEvaluated {len(df_3)} mixtures')
print(E.summarize(df_3).to_string(index=False))

### 5. Listen to Separated Audio

In [ ]:
import IPython.display as ipd
import random

# Play a random 3-speaker example
r = random.choice(rows_3)
mid = r['mixture_id']
print('MIXTURE (input):')
ipd.display(ipd.Audio(r['mixture_path']))

for k in range(1, N_SRC_3+1):
    print(f'estimate s{k}:')
    ipd.display(ipd.Audio(os.path.join(OUT_3, f'{mid}_s{k}.wav')))